En este cuaderno se realizará un análisis de datos para explorar la relación entre la población y las denuncias de violencia contra la mujer en Perú, utilizando un enfoque de *Data Mining*.

El objetivo principal es calcular la tasa de denuncias por cada 1000 habitantes a nivel distrital. Para ello, primero se procesará el dataset de población.

# Violencia contra la mujer - Data Mining

Unidad de análisis: un distrito del País

Variable de interés: Tasa de denuncias por cada 1000 habitantes en el distrito.

In [1]:
# importar librerias

import pandas as pd

### Carga y Preparación de Datos de Población

Este dataset contiene la población total proyectada al 30 de junio de cada año, según departamento, provincia y distrito, para el periodo 2018-2026. Es crucial para normalizar las cifras de denuncias por violencia, permitiendo calcular tasas por habitante.

## Dataset de población estimada en el país

In [2]:
poblacion = pd.read_excel('../data/raw/6894980-peru-poblacion-total-proyectada-al-30-de-junio-de-cada-ano-segun-departamento-provincia-y-distrito-2018-2026.xlsx', skiprows = 4)

poblacion.columns = ['UBIGEO', 'Nombre', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026'] # renombramos las columnas

La celda anterior carga el archivo Excel que contiene la data de población. Se saltan las primeras 4 filas (`skiprows = 4`) ya que contienen metadatos y encabezados no útiles para el DataFrame. Luego, se renombramos las columnas para facilitar su manipulación.

In [3]:
poblacion

,UBIGEO,Nombre,2018,2019,2020,2021,2022,2023,2024,2025,2026
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,000000,PERÚ,31562130,32131400,32625948,33035304,33396698,33725844,34038457,34350244.0,34660114.0
2,010000,AMAZONAS,419833,423863,426806,428512,429483,429943,430123,430251.0,430305.0
3,010100,CHACHAPOYAS,61078,62207,63188,63661,64058,64403,64718,65024.0,65320.0
4,010101,CHACHAPOYAS,36602,37870,39051,39691,40274,40817,41335,41843.0,42342.0
...,...,...,...,...,...,...,...,...,...,...,...
2135,19/ De acuerdo a la Ley N° 31644 de creación d...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2136,20/ Delimitación del distrito de Laredo y El P...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2137,"21/ En octubre de 2024, los distritos de Bella...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2138,22/ Según Ley Nº 32403 publicado en el diario ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Procesamiento y Limpieza de Datos de Población

Los siguientes pasos se enfocan en limpiar, transformar y estructurar el dataset de población para que sea adecuado para el análisis.

1.  **Limpieza de UBIGEOs y Nombres:** Se convierte la columna `UBIGEO` a numérica para identificar y eliminar filas de pie de página o metadatos que no corresponden a registros de población.
2.  **Creación de Columnas Jerárquicas:** Se identifican y extraen los nombres de `Departamento` y `Provincia` basándose en el formato del código UBIGEO, que sigue una jerarquía.
3.  **Filtrado por Distrito:** Se selecciona solo la información correspondiente a distritos (UBIGEOs que no terminan en '00') para el análisis a nivel distrital.
4.  **Transformación a Formato Largo (Melt):** Se utiliza `pd.melt` para transformar las columnas de años (2018-2026) en un formato más manejable, donde cada fila representa la población de un distrito en un año específico.
5.  **Ajuste de Tipos de Datos:** Se asegura que la columna `UBIGEO` tenga un formato de 6 dígitos rellenando con ceros a la izquierda y que la columna `Población` sea numérica.

In [4]:
# forzamos conversion a tipo de dato numérico, para obtener nan en pie de página
poblacion['UBIGEO_NUM'] = pd.to_numeric(poblacion['UBIGEO'], errors='coerce')
poblacion = poblacion.dropna(subset=['UBIGEO_NUM']).copy() # eliminar valores nan

# creamos columnas adicionales
poblacion['Departamento'] = None
poblacion['Provincia'] = None

# asignar el nombre de departamento para ubigeos divisibles entre 10000
poblacion.loc[poblacion['UBIGEO_NUM'] % 10000 == 0, 'Departamento'] = poblacion['Nombre']

# asignar el nombre de provincia para ubigeos divisibles  entre 100 pero no entre 10000
es_provincia = (poblacion['UBIGEO_NUM'] % 100 == 0) & (poblacion['UBIGEO_NUM'] % 10000 != 0)
poblacion.loc[es_provincia, 'Provincia'] = poblacion['Nombre']

# rellenar el nombre de departamento y provincia hacia abajo
poblacion['Departamento'] = poblacion['Departamento'].ffill()
poblacion['Provincia'] = poblacion['Provincia'].ffill()

## filtrar por distrito, no divisibles entre 100 y renombrar columna a Distrito
poblacion_distritos = poblacion[poblacion['UBIGEO_NUM'] % 100 != 0].copy()
poblacion_distritos = poblacion_distritos.rename(columns={'Nombre': 'Distrito'})

# transformar el los valores de "ancho" a "largo"
columnas_identificadoras = ['Departamento', 'Provincia', 'Distrito', 'UBIGEO']
años = ['2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026']

poblacion_final = pd.melt(
    poblacion_distritos,
    id_vars=columnas_identificadoras,
    value_vars=años,
    var_name='Año',
    value_name='Población'
)

# volver al ubigeo de 6 cifras, puede empezar en 0
poblacion_final['UBIGEO'] = pd.to_numeric(poblacion_final['UBIGEO']).astype(int).astype(str).str.zfill(6)

# convertir la columna Población a tipo de dato numérico
poblacion_final['Población'] = pd.to_numeric(poblacion_final['Población'], errors='coerce')

La celda anterior muestra un resumen de los tipos de datos y la cantidad de valores no nulos en el DataFrame `poblacion_final`. Es importante observar que la columna `población` tiene algunos valores nulos, lo cual se explica en la siguiente celda de texto.

In [5]:
poblacion_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 17028 entries, 0 to 17027
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Departamento  17028 non-null  str    
 1   Provincia     17028 non-null  str    
 2   Distrito      17028 non-null  str    
 3   UBIGEO        17028 non-null  str    
 4   Año           17028 non-null  str    
 5   Población     16969 non-null  float64
dtypes: float64(1), str(5)
memory usage: 798.3 KB


Esta celda proporciona estadísticas descriptivas básicas de la columna `Población`, como la media, la desviación estándar, el mínimo, el máximo y los cuartiles. Esto nos da una idea general de la distribución de la población en los distritos.

Existen valores nulos en `Población` debido a la existencia de distritos creados posteriormente al año base (2018).

In [6]:
poblacion_final.describe().round()

,Población
count,16969.0
mean,17651.0
std,58188.0
min,104.0
25%,1487.0
50%,3863.0
75%,10722.0
max,1300785.0


En esta celda, se realizan los siguientes pasos de limpieza y verificación final:

1.  **Nombres de columnas a minúsculas:** Se unifican los nombres de las columnas a minúsculas para facilitar su manejo y evitar errores de capitalización.
2.  **Verificación de forma y valores faltantes:** Se imprime el número de filas y columnas del DataFrame, así como la cantidad de valores faltantes por columna, para asegurar la integridad de los datos.
3.  **Identificación del distrito con mayor población:** Se muestra el registro del distrito con la población máxima, lo que puede ser útil para verificar la consistencia de los datos.

In [7]:
# nombres de las variables a minúsculas
poblacion_final.columns = poblacion_final.columns.str.lower()

In [8]:
print(f'El dataframe tiene {poblacion_final.shape[0]} filas y {poblacion_final.shape[1]} columnas, ')
print(f'Los valores faltantes por columna es:\n{poblacion_final.isna().sum()}')
print(f'El registro del distrito con mayor población es:\n{poblacion_final[poblacion_final.población == poblacion_final.población.max()]}')

El dataframe tiene 17028 filas y 6 columnas, 
Los valores faltantes por columna es:
departamento     0
provincia        0
distrito         0
ubigeo           0
año              0
población       59
dtype: int64
El registro del distrito con mayor población es:
      departamento           provincia                distrito  ubigeo   año  \
16460        LIMA   LIMA METROPOLITANA  SAN JUAN DE LURIGANCHO  150132  2026   

       población  
16460  1300785.0  


En la celda anterior se ajusta el formato del `UBIGEO` para asegurar que siempre tenga 6 dígitos, rellenando con ceros a la izquierda si es necesario, y se renombra la columna `año` a `anio` para consistencia.

In [9]:
poblacion_final['ubigeo'] = poblacion_final['ubigeo'].astype(str).str.zfill(6)
poblacion_final.rename(columns={'año': 'anio'}, inplace=True)

### Guardado de Datos Procesados

Finalmente, los datos de población procesados y limpios se guardan en un archivo CSV. Antes de guardar, se verifica la existencia del directorio de destino (`../data/processed`) y, si no existe, se crea para evitar errores. Este archivo `poblacion_por_distrito.csv` será utilizado en análisis posteriores.

In [10]:
import os

output_dir = '../data/processed'
if not os.path.exists(output_dir):
    os.makedirs(output_dir, exist_ok=True)

poblacion_final.to_csv(os.path.join(output_dir, 'poblacion_por_distrito.csv'), index=False)

In [11]:
poblacion_final

,departamento,provincia,distrito,ubigeo,anio,población
0,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,010101,2018,36602.0
1,AMAZONAS,CHACHAPOYAS,ASUNCION,010102,2018,283.0
2,AMAZONAS,CHACHAPOYAS,BALSAS,010103,2018,1222.0
3,AMAZONAS,CHACHAPOYAS,CHETO,010104,2018,687.0
4,AMAZONAS,CHACHAPOYAS,CHILIQUIN,010105,2018,619.0
...,...,...,...,...,...,...
17023,UCAYALI,PADRE ABAD,NESHUYA,250304,2026,12661.0
17024,UCAYALI,PADRE ABAD,ALEXANDER VON HUMBOLDT,250305,2026,6811.0
17025,UCAYALI,PADRE ABAD,HUIPOCA 6/,250306,2026,5010.0
17026,UCAYALI,PADRE ABAD,BOQUERON 11/,250307,2026,5887.0
